# Flight Price SQL + MLlib Starter
Notebook inicial para validar o ambiente PySpark, executar consultas SQL e treinar modelos basicos com pyspark.ml.
Diretrizes deste notebook:
- leitura priorizando Parquet no HDFS, com fallback para CSV apenas quando necessario;
- preparacao, validacao e exploracao usando spark.sql(...);
- modelagem com MLlib sobre uma base preparada integralmente em SQL;
- comparacao entre cenarios com e sem ase_fare;
- split temporal para reduzir vazamento entre treino e teste.


In [1]:
import os
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor, LinearRegression


In [2]:
APP_NAME = "flight-price-sql-mllib"
SPARK_MASTER_URL = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")
# Prefer the Parquet dataset (~6.8 GB, snappy-compressed, partitioned by startingAirport).
# Falls back to the raw CSV if Parquet is not yet generated. Run make convert-local
# from the project root to create the Parquet from the local CSV in one step.
HDFS_PARQUET_PATH  = "hdfs://namenode:9000/data/itineraries.parquet"
LOCAL_PARQUET_PATH = "/data/itineraries.parquet"
HDFS_CSV_PATH      = "hdfs://namenode:9000/data/itineraries.csv"
LOCAL_CSV_PATH     = "/data/itineraries.csv"
# With Parquet, 0.1 is a good default for iterative development. Increase to 1.0
# when you want a stronger benchmark and the cluster is stable.
LOAD_SAMPLE_RATIO = 0.1
DEV_SAMPLE_RATIO = 1.0
# Used only when reading the CSV fallback. Parquet carries its own typed schema.
RAW_SCHEMA = """
legId STRING,
searchDate STRING,
flightDate STRING,
startingAirport STRING,
destinationAirport STRING,
fareBasisCode STRING,
travelDuration STRING,
elapsedDays STRING,
isBasicEconomy STRING,
isRefundable STRING,
isNonStop STRING,
baseFare STRING,
totalFare STRING,
seatsRemaining STRING,
totalTravelDistance STRING,
segmentsDepartureTimeEpochSeconds STRING,
segmentsDepartureTimeRaw STRING,
segmentsArrivalTimeEpochSeconds STRING,
segmentsArrivalTimeRaw STRING,
segmentsArrivalAirportCode STRING,
segmentsDepartureAirportCode STRING,
segmentsAirlineName STRING,
segmentsAirlineCode STRING,
segmentsEquipmentDescription STRING,
segmentsDurationInSeconds STRING,
segmentsDistance STRING,
segmentsCabinCode STRING
"""
def get_or_create_spark(app_name: str = APP_NAME) -> SparkSession:
    spark_session = (
        SparkSession.builder
        .appName(app_name)
        .master(SPARK_MASTER_URL)
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
        .config("spark.sql.repl.eagerEval.enabled", "true")
        .config("spark.sql.repl.eagerEval.maxNumRows", "20")
        .config("spark.sql.repl.eagerEval.truncate", "80")
        .getOrCreate()
    )
    spark_session.sparkContext.setLogLevel("WARN")
    return spark_session
def resolve_candidate_paths(spark_session: SparkSession) -> list[tuple[str, str]]:
    master = spark_session.sparkContext.master
    paths: list[tuple[str, str]] = [(HDFS_PARQUET_PATH, "parquet")]
    if master.startswith("local"):
        paths += [
            (LOCAL_PARQUET_PATH, "parquet"),
            (HDFS_CSV_PATH, "csv"),
            (LOCAL_CSV_PATH, "csv"),
        ]
    else:
        paths += [(HDFS_CSV_PATH, "csv")]
    return paths
def load_flights(spark_session: SparkSession, candidate_paths: list[tuple[str, str]]):
    last_error = None
    print(f"Tentando carregar dataset pelos caminhos: {[p for p, _ in candidate_paths]}")
    for path, fmt in candidate_paths:
        try:
            if fmt == "parquet":
                df = spark_session.read.parquet(path)
            else:
                df = (
                    spark_session.read
                    .option("header", True)
                    .schema(RAW_SCHEMA)
                    .csv(path)
                )
            print(f"Dataset registrado ({fmt}): {path}")
            return df, path, fmt
        except Exception as exc:
            print(f"Falha ao carregar {path}: {exc}")
            last_error = exc
    raise RuntimeError("Nao foi possivel carregar o dataset em nenhum caminho") from last_error
def apply_load_sample(df, ratio: float):
    if ratio >= 1.0:
        print("Usando dataset completo apos a leitura.")
        return df
    print(f"Aplicando amostra de carga com ratio={ratio} para acelerar as iteracoes.")
    return df.sample(withReplacement=False, fraction=ratio, seed=42)
def run_sql(query: str, preview_rows: int = 20):
    df = spark.sql(query)
    display(df.limit(preview_rows).toPandas())
    return df


In [3]:
spark = get_or_create_spark()
candidate_paths = resolve_candidate_paths(spark)
raw_input_df, source_path, source_format = load_flights(spark, candidate_paths)
raw_df = apply_load_sample(raw_input_df, LOAD_SAMPLE_RATIO)
raw_df.createOrReplaceTempView("flights_raw")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Source path:", source_path)
print("Source format:", source_format)
print("Load sample ratio:", LOAD_SAMPLE_RATIO)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/02 02:28:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Tentando carregar dataset pelos caminhos: ['hdfs://namenode:9000/data/itineraries.parquet', 'hdfs://namenode:9000/data/itineraries.csv']


Dataset registrado (parquet): hdfs://namenode:9000/data/itineraries.parquet
Aplicando amostra de carga com ratio=0.1 para acelerar as iteracoes.
Spark version: 3.5.3
Spark master: spark://spark-master:7077
Source path: hdfs://namenode:9000/data/itineraries.parquet
Source format: parquet
Load sample ratio: 0.1


26/05/02 02:28:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
run_sql("SELECT * FROM flights_raw LIMIT 5")


,legId,searchDate,flightDate,destinationAirport,fareBasisCode,travelDuration,elapsedDays,isBasicEconomy,isRefundable,isNonStop,...,segmentsArrivalTimeRaw,segmentsArrivalAirportCode,segmentsDepartureAirportCode,segmentsAirlineName,segmentsAirlineCode,segmentsEquipmentDescription,segmentsDurationInSeconds,segmentsDistance,segmentsCabinCode,startingAirport
0,46d53f75fbb18f5ec60f8388931eda93,2022-06-22,2022-08-12,ATL,TAVQA0BQ,PT4H13M,1,True,False,True,...,2022-08-13T05:48:00.000-04:00,ATL,ONT,Delta,DL,Airbus A321,15180,1897,coach,LAX
1,76874575c3472c414afb153789558308,2022-06-23,2022-07-31,ATL,TNR,PT9H38M,0,False,False,False,...,2022-07-31T18:15:00.000-04:00||2022-07-31T23:2...,DTW||ATL,LAX||DTW,Spirit Airlines||Spirit Airlines,NK||NK,AIRBUS INDUSTRIE A320 SHARKLETS||,15900||7380,None||None,coach||coach,LAX
2,6e968f6b2d6485a81ac929b9167a8176,2022-06-23,2022-07-31,ATL,MA3NR,PT8H23M,0,False,False,False,...,2022-07-31T16:30:00.000-04:00||2022-07-31T19:2...,EWR||ATL,LAX||EWR,Spirit Airlines||Spirit Airlines,NK||NK,||Airbus A319,19680||8700,None||None,coach||coach,LAX
3,5fa07c18f7b1a09e18dac7d9010d6e14,2022-06-23,2022-07-31,ATL,VH7OAVMN,PT6H40M,0,False,False,False,...,2022-07-31T16:47:00.000-05:00||2022-07-31T21:1...,AUS||ATL,LAX||AUS,Alaska Airlines||Delta,AS||DL,Embraer 175||Airbus A321,11520||8400,1236||811,coach||coach,LAX
4,402caaa38047393ddb2437aaf867d491,2022-06-23,2022-07-31,ATL,QAVOA0MC,PT6H37M,0,False,False,False,...,2022-07-31T13:41:00.000-05:00||2022-07-31T18:0...,AUS||ATL,LAX||AUS,Delta||Delta,DL||DL,Airbus A320||Airbus A320,11460||8220,1236||811,coach||coach,LAX


legId,searchDate,flightDate,destinationAirport,fareBasisCode,travelDuration,elapsedDays,isBasicEconomy,isRefundable,isNonStop,baseFare,totalFare,seatsRemaining,totalTravelDistance,segmentsDepartureTimeEpochSeconds,segmentsDepartureTimeRaw,segmentsArrivalTimeEpochSeconds,segmentsArrivalTimeRaw,segmentsArrivalAirportCode,segmentsDepartureAirportCode,segmentsAirlineName,segmentsAirlineCode,segmentsEquipmentDescription,segmentsDurationInSeconds,segmentsDistance,segmentsCabinCode,startingAirport
46d53f75fbb18f5ec60f8388931eda93,2022-06-22,2022-08-12,ATL,TAVQA0BQ,PT4H13M,1,true,false,true,199.07,228.6,9,1897.0,1660368900,2022-08-12T22:35:00.000-07:00,1660384080,2022-08-13T05:48:00.000-04:00,ATL,ONT,Delta,DL,Airbus A321,15180,1897,coach,LAX
76874575c3472c414afb153789558308,2022-06-23,2022-07-31,ATL,TNR,PT9H38M,0,false,false,false,301.0,408.58,0,NULL,1659289800||1659317100,2022-07-31T10:50:00.000-07:00||2022-07-31T21:25:00.000-04:00,1659305700||1659324480,2022-07-31T18:15:00.000-04:00||2022-07-31T23:28:00.000-04:00,DTW||ATL,LAX||DTW,Spirit Airlines||Spirit Airlines,NK||NK,AIRBUS INDUSTRIE A320 SHARKLETS||,15900||7380,None||None,coach||coach,LAX
6e968f6b2d6485a81ac929b9167a8176,2022-06-23,2022-07-31,ATL,MA3NR,PT8H23M,0,false,false,false,312.0,419.58,0,NULL,1659279720||1659301200,2022-07-31T08:02:00.000-07:00||2022-07-31T17:00:00.000-04:00,1659299400||1659309900,2022-07-31T16:30:00.000-04:00||2022-07-31T19:25:00.000-04:00,EWR||ATL,LAX||EWR,Spirit Airlines||Spirit Airlines,NK||NK,||Airbus A319,19680||8700,None||None,coach||coach,LAX
5fa07c18f7b1a09e18dac7d9010d6e14,2022-06-23,2022-07-31,ATL,VH7OAVMN,PT6H40M,0,false,false,false,456.74,514.6,7,2047.0,1659292500||1659308100,2022-07-31T11:35:00.000-07:00||2022-07-31T17:55:00.000-05:00,1659304020||1659316500,2022-07-31T16:47:00.000-05:00||2022-07-31T21:15:00.000-04:00,AUS||ATL,LAX||AUS,Alaska Airlines||Delta,AS||DL,Embraer 175||Airbus A321,11520||8400,1236||811,coach||coach,LAX
402caaa38047393ddb2437aaf867d491,2022-06-23,2022-07-31,ATL,QAVOA0MC,PT6H37M,0,false,false,false,472.55,531.59,8,2047.0,1659281400||1659297000,2022-07-31T08:30:00.000-07:00||2022-07-31T14:50:00.000-05:00,1659292860||1659305220,2022-07-31T13:41:00.000-05:00||2022-07-31T18:07:00.000-04:00,AUS||ATL,LAX||AUS,Delta||Delta,DL||DL,Airbus A320||Airbus A320,11460||8220,1236||811,coach||coach,LAX


In [5]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_clean AS
WITH base AS (
    SELECT
        legId AS leg_id,
        TO_DATE(searchDate) AS search_date,
        TO_DATE(flightDate) AS flight_date,
        startingAirport AS starting_airport,
        destinationAirport AS destination_airport,
        fareBasisCode AS fare_basis_code,
        UPPER(SUBSTRING(COALESCE(fareBasisCode, 'UNK'), 1, 1)) AS fare_basis_prefix,
        travelDuration AS travel_duration_iso,
        CAST(elapsedDays AS INT) AS overnight_days,
        DATEDIFF(TO_DATE(flightDate), TO_DATE(searchDate)) AS days_until_flight,
        CASE WHEN LOWER(CAST(isBasicEconomy AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_basic_economy,
        CASE WHEN LOWER(CAST(isRefundable AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_refundable,
        CASE WHEN LOWER(CAST(isNonStop AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_non_stop,
        CAST(baseFare AS DOUBLE) AS base_fare,
        CAST(totalFare AS DOUBLE) AS total_fare,
        CAST(seatsRemaining AS INT) AS seats_remaining,
        CAST(totalTravelDistance AS DOUBLE) AS total_travel_distance,
        (
            CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)D', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)D', 1) END AS INT) * 1440
            + CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)H', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)H', 1) END AS INT) * 60
            + CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)M', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)M', 1) END AS INT)
        ) AS travel_duration_minutes,
        CAST(NULLIF(REGEXP_EXTRACT(COALESCE(segmentsDepartureTimeRaw, ''), 'T([0-9]{2}):', 1), '') AS INT) AS departure_hour,
        COALESCE(segmentsDepartureAirportCode, '') AS segments_departure_airport_code_raw,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsDepartureAirportCode, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS departure_airport_segments,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsDurationInSeconds, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS duration_seconds_segments,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsDistance, ''), '||', '~'), '~'), x -> TRIM(x) <> '' AND LOWER(TRIM(x)) <> 'none') AS distance_segments,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsAirlineCode, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS airline_segments,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsCabinCode, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS cabin_segments,
        DAYOFWEEK(TO_DATE(flightDate)) AS flight_day_of_week,
        MONTH(TO_DATE(flightDate)) AS flight_month,
        CONCAT(startingAirport, '-', destinationAirport) AS route
    FROM flights_raw
    WHERE totalFare IS NOT NULL
), prepared AS (
    SELECT
        *,
        CASE
            WHEN segments_departure_airport_code_raw IS NULL OR TRIM(segments_departure_airport_code_raw) = '' THEN 0
            ELSE CAST(((LENGTH(segments_departure_airport_code_raw) - LENGTH(REPLACE(segments_departure_airport_code_raw, '||', ''))) / 2) + 1 AS INT)
        END AS segment_count,
        AGGREGATE(duration_seconds_segments, CAST(0.0 AS DOUBLE), (acc, x) -> acc + COALESCE(CAST(x AS DOUBLE), 0.0D)) / 60.0 AS segment_duration_sum_minutes,
        AGGREGATE(distance_segments, CAST(0.0 AS DOUBLE), (acc, x) -> acc + COALESCE(CAST(x AS DOUBLE), 0.0D)) AS known_segment_distance,
        SIZE(distance_segments) AS known_segment_distance_count,
        SIZE(ARRAY_DISTINCT(airline_segments)) AS distinct_airline_count,
        SIZE(ARRAY_DISTINCT(cabin_segments)) AS distinct_cabin_count
    FROM base
), engineered AS (
    SELECT
        *,
        GREATEST(segment_count - 1, 0) AS stop_count,
        CASE WHEN segment_count > known_segment_distance_count THEN 1 ELSE 0 END AS has_missing_segment_distance,
        CASE WHEN segment_count > 0 THEN known_segment_distance / segment_count END AS distance_per_segment,
        CASE WHEN segment_count > 0 THEN segment_duration_sum_minutes / segment_count END AS segment_duration_avg_minutes,
        CASE
            WHEN total_travel_distance IS NULL OR isnan(total_travel_distance) THEN known_segment_distance
            ELSE total_travel_distance
        END AS effective_distance,
        GREATEST(travel_duration_minutes - segment_duration_sum_minutes, 0.0D) AS travel_minus_segment_minutes,
        CASE
            WHEN segment_count <= 1 THEN 0.0D
            ELSE GREATEST(travel_duration_minutes - segment_duration_sum_minutes, 0.0D)
        END AS layover_minutes,
        CASE WHEN flight_day_of_week IN (1, 7) THEN 1 ELSE 0 END AS is_weekend
    FROM prepared
)
SELECT
    leg_id,
    search_date,
    flight_date,
    starting_airport,
    destination_airport,
    fare_basis_code,
    fare_basis_prefix,
    travel_duration_iso,
    overnight_days,
    days_until_flight,
    is_basic_economy,
    is_refundable,
    is_non_stop,
    base_fare,
    total_fare,
    seats_remaining,
    total_travel_distance,
    travel_duration_minutes,
    departure_hour,
    segment_count,
    stop_count,
    segment_duration_sum_minutes,
    segment_duration_avg_minutes,
    known_segment_distance,
    known_segment_distance_count,
    effective_distance,
    distance_per_segment,
    layover_minutes,
    travel_minus_segment_minutes,
    has_missing_segment_distance,
    distinct_airline_count,
    distinct_cabin_count,
    flight_day_of_week,
    flight_month,
    is_weekend,
    route
FROM engineered
""")
if DEV_SAMPLE_RATIO < 1.0:
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW flights_dev AS
    SELECT *
    FROM flights_clean
    WHERE rand(42) <= {DEV_SAMPLE_RATIO}
    """)
else:
    spark.sql("CREATE OR REPLACE TEMP VIEW flights_dev AS SELECT * FROM flights_clean")


In [6]:
run_sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN days_until_flight < 0 THEN 1 ELSE 0 END) AS invalid_days_until_flight,
    SUM(CASE WHEN segment_count <= 0 THEN 1 ELSE 0 END) AS invalid_segment_count,
    SUM(CASE WHEN segment_count > 4 THEN 1 ELSE 0 END) AS invalid_segment_upper_bound,
    SUM(CASE WHEN travel_duration_minutes <= 0 THEN 1 ELSE 0 END) AS invalid_travel_duration,
    SUM(CASE WHEN departure_hour IS NOT NULL AND (departure_hour < 0 OR departure_hour > 23) THEN 1 ELSE 0 END) AS invalid_departure_hour,
    SUM(CASE WHEN layover_minutes < 0 THEN 1 ELSE 0 END) AS negative_layover_signal,
    SUM(CASE WHEN is_non_stop = 1 AND stop_count <> 0 THEN 1 ELSE 0 END) AS invalid_non_stop_stop_count,
    SUM(CASE WHEN is_non_stop = 1 AND layover_minutes > 15 THEN 1 ELSE 0 END) AS non_stop_with_layover,
    MIN(total_fare) AS min_total_fare,
    MAX(total_fare) AS max_total_fare,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(travel_duration_minutes), 2) AS avg_duration_minutes,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(layover_minutes), 2) AS avg_layover_minutes
FROM flights_dev
""")


,total_rows,invalid_days_until_flight,invalid_segment_count,invalid_segment_upper_bound,invalid_travel_duration,invalid_departure_hour,negative_layover_signal,invalid_non_stop_stop_count,non_stop_with_layover,min_total_fare,max_total_fare,avg_total_fare,avg_duration_minutes,avg_days_until_flight,avg_layover_minutes
0,8213944,0,0,11,0,0,0,0,0,19.59,8260.61,340.47,428.27,26.9,143.71


total_rows,invalid_days_until_flight,invalid_segment_count,invalid_segment_upper_bound,invalid_travel_duration,invalid_departure_hour,negative_layover_signal,invalid_non_stop_stop_count,non_stop_with_layover,min_total_fare,max_total_fare,avg_total_fare,avg_duration_minutes,avg_days_until_flight,avg_layover_minutes
8213944,0,0,11,0,0,0,0,0,19.59,8260.61,340.47,428.27,26.9,143.71


In [7]:
run_sql("""
SELECT
    segment_count,
    stop_count,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(travel_duration_minutes), 2) AS avg_duration_minutes,
    ROUND(AVG(layover_minutes), 2) AS avg_layover_minutes,
    ROUND(AVG(distinct_airline_count), 2) AS avg_distinct_airlines
FROM flights_dev
GROUP BY segment_count, stop_count
ORDER BY segment_count, stop_count
LIMIT 20
""")


,segment_count,stop_count,total_voos,avg_total_fare,avg_duration_minutes,avg_layover_minutes,avg_distinct_airlines
0,1,0,2207415,251.17,181.11,0.00,1.00
1,2,1,5228208,346.64,480.91,173.09,1.08
2,3,2,758282,550.15,777.77,354.25,1.60
3,4,3,20028,635.38,694.86,342.45,2.10
4,5,4,11,835.90,1564.36,803.09,2.00


segment_count,stop_count,total_voos,avg_total_fare,avg_duration_minutes,avg_layover_minutes,avg_distinct_airlines
1,0,2207415,251.17,181.11,0.0,1.0
2,1,5228208,346.64,480.91,173.09,1.08
3,2,758282,550.15,777.77,354.25,1.6
4,3,20028,635.38,694.86,342.45,2.1
5,4,11,835.9,1564.36,803.09,2.0


In [8]:
run_sql("""
SELECT
    route,
    travel_duration_iso,
    travel_duration_minutes,
    segment_count,
    stop_count,
    departure_hour,
    layover_minutes,
    days_until_flight,
    total_fare
FROM flights_dev
WHERE travel_duration_minutes <= 0
   OR segment_count <= 0
   OR days_until_flight < 0
   OR (departure_hour IS NOT NULL AND (departure_hour < 0 OR departure_hour > 23))
ORDER BY total_fare DESC
LIMIT 20
""")


,route,travel_duration_iso,travel_duration_minutes,segment_count,stop_count,departure_hour,layover_minutes,days_until_flight,total_fare


route,travel_duration_iso,travel_duration_minutes,segment_count,stop_count,departure_hour,layover_minutes,days_until_flight,total_fare


In [9]:
run_sql("""
SELECT
    route,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(base_fare), 2) AS avg_base_fare
FROM flights_dev
GROUP BY route
ORDER BY avg_total_fare DESC
LIMIT 20
""")


,route,total_voos,avg_total_fare,avg_base_fare
0,OAK-MIA,18269,664.57,587.63
1,IAD-OAK,16774,659.01,585.79
2,LGA-OAK,40174,658.18,584.28
3,OAK-LGA,38010,654.02,580.45
4,OAK-CLT,20279,648.86,573.96
5,OAK-BOS,26510,642.29,569.82
6,MIA-OAK,15742,637.36,559.79
7,PHL-OAK,22546,624.85,551.50
8,CLT-OAK,20085,620.79,547.69
9,OAK-PHL,25444,618.61,544.75


route,total_voos,avg_total_fare,avg_base_fare
OAK-MIA,18269,664.57,587.63
IAD-OAK,16774,659.01,585.79
LGA-OAK,40174,658.18,584.28
OAK-LGA,38010,654.02,580.45
OAK-CLT,20279,648.86,573.96
OAK-BOS,26510,642.29,569.82
MIA-OAK,15742,637.36,559.79
PHL-OAK,22546,624.85,551.5
CLT-OAK,20085,620.79,547.69
OAK-PHL,25444,618.61,544.75


In [10]:
run_sql("""
SELECT
    is_non_stop,
    stop_count,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(layover_minutes), 2) AS avg_layover_minutes
FROM flights_dev
GROUP BY is_non_stop, stop_count
ORDER BY is_non_stop DESC, stop_count ASC
""")


,is_non_stop,stop_count,total_voos,avg_total_fare,avg_days_until_flight,avg_layover_minutes
0,1,0,2207415,251.17,27.28,0.00
1,0,1,5228208,346.64,26.75,173.09
2,0,2,758282,550.15,26.84,354.25
3,0,3,20028,635.38,26.39,342.45
4,0,4,11,835.90,32.00,803.09


is_non_stop,stop_count,total_voos,avg_total_fare,avg_days_until_flight,avg_layover_minutes
1,0,2207415,251.17,27.28,0.0
0,1,5228208,346.64,26.75,173.09
0,2,758282,550.15,26.84,354.25
0,3,20028,635.38,26.39,342.45
0,4,11,835.9,32.0,803.09


In [11]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_ml_base AS
SELECT
    total_fare,
    base_fare,
    days_until_flight,
    overnight_days,
    seats_remaining,
    COALESCE(effective_distance, 0.0D) AS total_travel_distance,
    travel_duration_minutes,
    segment_count,
    stop_count,
    flight_day_of_week,
    flight_month,
    is_basic_economy,
    is_refundable,
    is_non_stop,
    route,
    flight_date,
    starting_airport,
    destination_airport,
    fare_basis_prefix,
    departure_hour,
    segment_duration_sum_minutes,
    segment_duration_avg_minutes,
    effective_distance,
    distance_per_segment,
    layover_minutes,
    has_missing_segment_distance,
    distinct_airline_count,
    distinct_cabin_count,
    is_weekend,
    travel_minus_segment_minutes
FROM flights_dev
WHERE total_fare IS NOT NULL
  AND base_fare IS NOT NULL
  AND days_until_flight IS NOT NULL
  AND overnight_days IS NOT NULL
  AND seats_remaining IS NOT NULL
  AND travel_duration_minutes IS NOT NULL
  AND departure_hour IS NOT NULL
  AND segment_count BETWEEN 1 AND 4
  AND departure_hour BETWEEN 0 AND 23
  AND travel_duration_minutes BETWEEN 30 AND 4320
  AND segment_duration_sum_minutes > 0
  AND days_until_flight BETWEEN 0 AND 365
  AND layover_minutes BETWEEN 0 AND 1440
  AND (is_non_stop = 0 OR stop_count = 0)
  AND (is_non_stop = 0 OR layover_minutes <= 15)
""")
run_sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(flight_date) AS min_flight_date,
    MAX(flight_date) AS max_flight_date,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(segment_count), 2) AS avg_segment_count,
    ROUND(AVG(layover_minutes), 2) AS avg_layover_minutes
FROM flights_ml_base
""")
cutoff_unix = spark.sql("SELECT percentile_approx(unix_timestamp(flight_date), 0.8) AS cutoff_unix FROM flights_ml_base").first()["cutoff_unix"]
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW flights_train AS
SELECT *
FROM flights_ml_base
WHERE unix_timestamp(flight_date) <= {cutoff_unix}
""")
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW flights_test AS
SELECT *
FROM flights_ml_base
WHERE unix_timestamp(flight_date) > {cutoff_unix}
""")
train_df = spark.sql("SELECT * FROM flights_train")
test_df = spark.sql("SELECT * FROM flights_test")
train_df.cache()
test_df.cache()
run_sql("""
SELECT 'train' AS split_name, COUNT(*) AS total_rows, MIN(flight_date) AS min_flight_date, MAX(flight_date) AS max_flight_date FROM flights_train
UNION ALL
SELECT 'test' AS split_name, COUNT(*) AS total_rows, MIN(flight_date) AS min_flight_date, MAX(flight_date) AS max_flight_date FROM flights_test
""")
print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


,total_rows,min_flight_date,max_flight_date,avg_total_fare,avg_days_until_flight,avg_segment_count,avg_layover_minutes
0,8213835,2022-04-17,2022-11-19,340.47,26.9,1.83,143.69


,split_name,total_rows,min_flight_date,max_flight_date
0,train,6580886,2022-04-17,2022-09-25
1,test,1632949,2022-09-26,2022-11-19


Train rows: 6580886


[Stage 58:=====================================================>(167 + 2) / 169]

Test rows: 1632949


In [12]:
COMMON_FEATURE_COLS = [
    "days_until_flight",
    "overnight_days",
    "seats_remaining",
    "total_travel_distance",
    "travel_duration_minutes",
    "segment_count",
    "stop_count",
    "flight_day_of_week",
    "flight_month",
    "is_basic_economy",
    "is_refundable",
    "is_non_stop",
    "departure_hour",
    "segment_duration_sum_minutes",
    "segment_duration_avg_minutes",
    "effective_distance",
    "distance_per_segment",
    "layover_minutes",
    "has_missing_segment_distance",
    "distinct_airline_count",
    "distinct_cabin_count",
    "is_weekend",
    "travel_minus_segment_minutes"
]
CATEGORICAL_FEATURE_COLS = [
    "route",
    "starting_airport",
    "destination_airport",
    "fare_basis_prefix"
]
FEATURE_SETS = {
    "with_base_fare": ["base_fare"] + COMMON_FEATURE_COLS,
    "without_base_fare": COMMON_FEATURE_COLS
}
regression_evaluators = {
    "rmse": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="rmse"),
    "mae": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="mae"),
    "r2": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="r2")
}
train_count = train_df.count()
test_count = test_df.count()
def build_pipeline(feature_cols, estimator):
    indexers = []
    index_output_cols = []
    encoded_output_cols = []
    for column_name in CATEGORICAL_FEATURE_COLS:
        index_col = f"{column_name}_index"
        encoded_col = f"{column_name}_ohe"
        indexers.append(StringIndexer(inputCol=column_name, outputCol=index_col, handleInvalid="keep"))
        index_output_cols.append(index_col)
        encoded_output_cols.append(encoded_col)
    encoder = OneHotEncoder(inputCols=index_output_cols, outputCols=encoded_output_cols)
    assembler = VectorAssembler(inputCols=feature_cols + encoded_output_cols, outputCol="features")
    return Pipeline(stages=indexers + [encoder, assembler, estimator])
def train_and_evaluate(model_name: str, estimator, feature_set_name: str, feature_cols):
    pipeline = build_pipeline(feature_cols, estimator)
    fitted_pipeline = pipeline.fit(train_df)
    predictions = fitted_pipeline.transform(test_df)
    metrics = {
        "model": model_name,
        "feature_set": feature_set_name,
        "uses_base_fare": 1 if "base_fare" in feature_cols else 0,
        "rmse": regression_evaluators["rmse"].evaluate(predictions),
        "mae": regression_evaluators["mae"].evaluate(predictions),
        "r2": regression_evaluators["r2"].evaluate(predictions),
        "train_rows": train_count,
        "test_rows": test_count
    }
    return fitted_pipeline, predictions, metrics


In [13]:
model_runs = {}
results = []
for feature_set_name, feature_cols in FEATURE_SETS.items():
    linear_regression = LinearRegression(
        featuresCol="features",
        labelCol="total_fare",
        predictionCol="prediction",
        maxIter=20,
        regParam=0.1,
        elasticNetParam=0.0
    )
    decision_tree = DecisionTreeRegressor(
        featuresCol="features",
        labelCol="total_fare",
        predictionCol="prediction",
        maxDepth=10,
        minInstancesPerNode=100
    )
    for base_model_name, estimator in [
        ("linear_regression", linear_regression),
        ("decision_tree", decision_tree)
    ]:
        run_name = f"{base_model_name}__{feature_set_name}"
        fitted_pipeline, predictions, metrics = train_and_evaluate(run_name, estimator, feature_set_name, feature_cols)
        model_runs[run_name] = {
            "pipeline": fitted_pipeline,
            "predictions": predictions,
            "metrics": metrics
        }
        results.append(metrics)
metrics_df = spark.createDataFrame(results)
display(metrics_df.orderBy("rmse").toPandas())


26/05/02 02:35:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/02 02:35:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/05/02 02:35:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
                                                                                

,feature_set,mae,model,r2,rmse,test_rows,train_rows,uses_base_fare
0,with_base_fare,3.670453,linear_regression__with_base_fare,0.998567,6.879973,1632949,6580886,1
1,with_base_fare,7.127913,decision_tree__with_base_fare,0.980621,25.304079,1632949,6580886,1
2,without_base_fare,71.124809,decision_tree__without_base_fare,0.691720,100.923374,1632949,6580886,0
3,without_base_fare,77.197592,linear_regression__without_base_fare,0.655728,106.652302,1632949,6580886,0


In [14]:
best_run_name = metrics_df.orderBy("rmse").first()["model"]
best_no_base_run_name = metrics_df.filter("uses_base_fare = 0").orderBy("rmse").first()["model"]
best_predictions = model_runs[best_run_name]["predictions"]
best_predictions.createOrReplaceTempView("best_predictions")
best_no_base_predictions = model_runs[best_no_base_run_name]["predictions"]
best_no_base_predictions.createOrReplaceTempView("best_no_base_predictions")
print("Best overall run:", best_run_name)
print("Best no-base-fare run:", best_no_base_run_name)
run_sql("""
SELECT
    stop_count,
    COUNT(*) AS total_voos,
    ROUND(AVG(ABS(total_fare - prediction)), 2) AS avg_absolute_error,
    ROUND(MAX(ABS(total_fare - prediction)), 2) AS max_absolute_error,
    ROUND(AVG(total_fare), 2) AS avg_total_fare
FROM best_no_base_predictions
GROUP BY stop_count
ORDER BY stop_count
""")
run_sql("""
SELECT
    route,
    total_fare,
    ROUND(prediction, 2) AS prediction,
    ROUND(ABS(total_fare - prediction), 2) AS absolute_error,
    days_until_flight,
    travel_duration_minutes,
    layover_minutes,
    segment_count,
    stop_count,
    is_non_stop,
    distinct_airline_count,
    fare_basis_prefix
FROM best_no_base_predictions
ORDER BY absolute_error DESC
LIMIT 20
""")


Best overall run: linear_regression__with_base_fare
Best no-base-fare run: decision_tree__without_base_fare


,stop_count,total_voos,avg_absolute_error,max_absolute_error,avg_total_fare
0,0,399738,60.90,4543.25,206.80
1,1,1045758,72.69,6719.41,289.26
2,2,184263,84.30,2098.92,519.44
3,3,3190,79.13,693.24,589.03


,route,total_fare,prediction,absolute_error,days_until_flight,travel_duration_minutes,layover_minutes,segment_count,stop_count,is_non_stop,distinct_airline_count,fare_basis_prefix
0,DTW-OAK,7918.60,1199.19,6719.41,32,493,64.0,2,1,0,1,Y
1,DTW-OAK,7918.60,1199.19,6719.41,27,493,64.0,2,1,0,1,Y
2,DTW-SFO,7548.60,1199.19,6349.41,25,534,104.0,2,1,0,1,Y
3,SFO-DTW,7548.60,1199.19,6349.41,26,469,65.0,2,1,0,1,Y
4,SFO-DTW,7548.60,1199.19,6349.41,29,469,65.0,2,1,0,1,Y
5,SFO-DTW,7548.60,1199.19,6349.41,25,469,65.0,2,1,0,1,Y
6,BOS-LAX,4923.60,380.35,4543.25,23,383,0.0,1,0,1,1,F
7,SFO-BOS,3422.60,380.35,3042.25,19,482,65.0,2,1,0,1,C
8,LAX-CLT,3250.20,380.35,2869.85,23,734,291.0,2,1,0,1,C
9,LAX-CLT,3250.20,380.35,2869.85,48,734,291.0,2,1,0,1,C


route,total_fare,prediction,absolute_error,days_until_flight,travel_duration_minutes,layover_minutes,segment_count,stop_count,is_non_stop,distinct_airline_count,fare_basis_prefix
DTW-OAK,7918.6,1199.19,6719.41,32,493,64.0,2,1,0,1,Y
DTW-OAK,7918.6,1199.19,6719.41,27,493,64.0,2,1,0,1,Y
DTW-SFO,7548.6,1199.19,6349.41,25,534,104.0,2,1,0,1,Y
SFO-DTW,7548.6,1199.19,6349.41,26,469,65.0,2,1,0,1,Y
SFO-DTW,7548.6,1199.19,6349.41,29,469,65.0,2,1,0,1,Y
SFO-DTW,7548.6,1199.19,6349.41,25,469,65.0,2,1,0,1,Y
BOS-LAX,4923.6,380.35,4543.25,23,383,0.0,1,0,1,1,F
SFO-BOS,3422.6,380.35,3042.25,19,482,65.0,2,1,0,1,C
LAX-CLT,3250.2,380.35,2869.85,23,734,291.0,2,1,0,1,C
LAX-CLT,3250.2,380.35,2869.85,48,734,291.0,2,1,0,1,C


## Proximos passos
- gerar hdfs://namenode:9000/data/itineraries.parquet com make convert-local e priorizar esse caminho para os proximos reruns;
- subir LOAD_SAMPLE_RATIO para 1.0 quando quiser benchmark mais forte e o cluster estiver estavel;
- manter a leitura with_base_fare como baseline de teto e usar without_base_fare como avaliacao mais realista;
- testar RandomForestRegressor e GBTRegressor como proximos baselines quando o pipeline estiver consolidado.


In [15]:
print("Notebook pronto para exploracao interativa no Jupyter Lab.")


Notebook pronto para exploracao interativa no Jupyter Lab.
